# Practical 09: Missing Value Imputation with SimpleImputer & Categorical Encoding

**UNIT 3: Preparing Big Data for Machine Learning**

**Problem Statement:** Write a Python program using Scikit-Learn to preprocess a machine learning dataset: handle missing numerical values using SimpleImputer with the mean strategy, transform categorical text labels into ordinal integers using LabelEncoder, and convert nominal variables to binary dummy indicators using One-Hot Encoding.

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

rnd = np.random.default_rng(123)
n = 48
age = rnd.integers(24, 62, n).astype(float)
salary = np.round(30000 + age * 1010 + rnd.normal(0, 7500, n), -2)
tbl = pd.DataFrame({
    "Country": rnd.choice(["Greece", "Peru", "Nepal"], n),
    "Age": age,
    "Salary": salary,
    "Purchased": np.where(salary + rnd.normal(0, 8000, n) > 57000, "Yes", "No"),
})
tbl.loc[rnd.choice(n, 6, replace=False), "Age"] = np.nan
tbl.loc[rnd.choice(n, 6, replace=False), "Salary"] = np.nan
print(tbl.head(12))

print(tbl.isna().sum())

num_imputer = SimpleImputer(missing_values=np.nan, strategy="median")
tbl[["Age", "Salary"]] = num_imputer.fit_transform(tbl[["Age", "Salary"]])

print("Learned medians (Age, Salary):", num_imputer.statistics_.round(2))
print("\nMissing values after imputation:")
print(tbl.isna().sum())

le_target = LabelEncoder()
tbl["Purchased_Encoded"] = le_target.fit_transform(tbl["Purchased"])
print("Purchased mapping:", dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))

tbl["Country_Label"] = LabelEncoder().fit_transform(tbl["Country"])
print("\nNote: Country_Label implies an artificial order (Greece < Nepal < Peru),")
print("so LabelEncoder is unsuitable for nominal features in linear models.")
print(tbl[["Country", "Country_Label", "Purchased", "Purchased_Encoded"]].head())

oh_enc = OneHotEncoder(sparse_output=False, dtype=int)
encoded = oh_enc.fit_transform(tbl[["Country"]])
ohe_df = pd.DataFrame(encoded, columns=oh_enc.get_feature_names_out(["Country"]), index=tbl.index)

out_df = pd.concat([tbl.drop(columns=["Country", "Country_Label"]), ohe_df], axis=1)
print(out_df.head(12))

print(pd.get_dummies(tbl["Country"], prefix="Country", dtype=int).head())

,Country,Age,Salary,Purchased
0,Nepal,NaN,59900.0,Yes
1,Peru,NaN,78400.0,Yes
2,Peru,46.0,86100.0,Yes
3,Peru,26.0,64300.0,Yes
4,Greece,58.0,91500.0,Yes
5,Nepal,32.0,62400.0,Yes
6,Greece,33.0,60600.0,Yes
7,Peru,31.0,52100.0,No
8,Greece,36.0,NaN,Yes
9,Greece,30.0,44000.0,No


Country      0
Age          6
Salary       6
Purchased    0
dtype: int64
Learned medians (Age, Salary): [3.950e+01 6.985e+04]

Missing values after imputation:
Country      0
Age          0
Salary       0
Purchased    0
dtype: int64
Purchased mapping: {'No': np.int64(0), 'Yes': np.int64(1)}

Note: Country_Label implies an artificial order (Greece < Nepal < Peru),
so LabelEncoder is unsuitable for nominal features in linear models.

,Country,Country_Label,Purchased,Purchased_Encoded
0,Nepal,1,Yes,1
1,Peru,2,Yes,1
2,Peru,2,Yes,1
3,Peru,2,Yes,1
4,Greece,0,Yes,1


,Age,Salary,Purchased,Purchased_Encoded,Country_Greece,Country_Nepal,Country_Peru
0,39.5,59900.0,Yes,1,0,1,0
1,39.5,78400.0,Yes,1,0,0,1
2,46.0,86100.0,Yes,1,0,0,1
3,26.0,64300.0,Yes,1,0,0,1
4,58.0,91500.0,Yes,1,1,0,0
5,32.0,62400.0,Yes,1,0,1,0
6,33.0,60600.0,Yes,1,1,0,0
7,31.0,52100.0,No,0,0,0,1
8,36.0,69850.0,Yes,1,1,0,0
9,30.0,44000.0,No,0,1,0,0


,Country_Greece,Country_Nepal,Country_Peru
0,0,1,0
1,0,0,1
2,0,0,1
3,0,0,1
4,1,0,0
